In [ ]:
from matplotlib import pyplot as plt

def menu():
    print("""
******** MENU METEOROLÓGICO ********
1: Calcular a temperatura média de cada dia
2: Guardar a tabela meteorológica num ficheiro
3: Carregar uma tabela meteorológica de um ficheiro
4: Calcular a temperatura mínima mais baixa
5: Calcular a amplitude térmica de cada dia
6: Encontrar o dia com precipitação máxima
7: Listar dias com precipitação superior a um limite
8: Calcular o maior período consecutivo com precipitação abaixo de um limite
9: Desenhar gráficos da temperatura mínima, máxima e pluviosidade
0: Sair da aplicação
""")

def dataTexto(data):
    ano, mes, dia = data
    return f"{ano}-{mes}-{dia}"

def medias(tabMeteo):
    res = []
    for data, tmin, tmax, prec in tabMeteo:
        res.append((data, (tmin + tmax) / 2))
    return res

def guardaTabMeteo(tabMeteo, fnome):
    file = open(fnome, "w")
    for data, tmin, tmax, prec in tabMeteo:
        ano, mes, dia = data
        file.write(f"{ano}/{mes}/{dia}|{tmin}|{tmax}|{prec}\n")
    file.close()

def carregaTabMeteo(fnome):
    res = []
    file = open(fnome, "r")
    for linha in file:
        linhaLimpa = linha.strip()
        if len(linhaLimpa) > 0 and "Data" not in linhaLimpa:
            partes1 = linhaLimpa.split("|")
            if len(partes1) == 4:
                partes2 = partes1[0].split("/")
                ano = partes2[0]
                mes = partes2[1]
                dia = partes2[2]
                tmin = partes1[1]
                tmax = partes1[2]
                prec = partes1[3]
                texto = ((int(ano), int(mes), int(dia)), float(tmin), float(tmax), float(prec))
                res.append(texto)
    file.close()
    return res

def minMin(tabMeteo):
    if len(tabMeteo) == 0:
        return None
    minima = tabMeteo[0][1]
    for data, tmin, tmax, prec in tabMeteo:
        if tmin < minima:
            minima = tmin
    return minima

def amplTerm(tabMeteo):
    res = []
    for data, tmin, tmax, prec in tabMeteo:
        res.append((data, tmax - tmin))
    return res

def maxChuva(tabMeteo):
    if len(tabMeteo) == 0:
        return None
    max_data = tabMeteo[0][0]
    max_prec = tabMeteo[0][3]
    for data, tmin, tmax, prec in tabMeteo:
        if prec > max_prec:
            max_data = data
            max_prec = prec
    return (max_data, max_prec)

def diasChuvosos(tabMeteo, p):
    res = []
    for data, tmin, tmax, prec in tabMeteo:
        if prec > p:
            res.append((data, prec))
    return res

def maxPeriodoCalor(tabMeteo, p):
    maior = 0
    atual = 0
    for data, tmin, tmax, prec in tabMeteo:
        if prec < p:
            atual = atual + 1
            if atual > maior:
                maior = atual
        else:
            atual = 0
    return maior

def grafTabMeteo(tabMeteo):
    if len(tabMeteo) == 0:
        print("A tabela meteorológica está vazia.")
    else:
        x = [dataTexto(data) for data, tmin, tmax, prec in tabMeteo]
        y_min = [tmin for data, tmin, tmax, prec in tabMeteo]
        y_max = [tmax for data, tmin, tmax, prec in tabMeteo]
        y_prec = [prec for data, tmin, tmax, prec in tabMeteo]

        plt.figure()
        plt.plot(x, y_min, marker="o", label="Temperatura mínima (ºC)")
        plt.plot(x, y_max, marker="o", label="Temperatura máxima (ºC)")
        plt.title("Temperaturas por dia")
        plt.xlabel("Data")
        plt.ylabel("Temperatura (ºC)")
        plt.legend()
        plt.grid()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        plt.figure()
        plt.bar(x, y_prec, label="Precipitação (mm)")
        plt.title("Precipitação por dia")
        plt.xlabel("Data")
        plt.ylabel("Precipitação (mm)")
        plt.legend()
        plt.grid(axis="y")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

def mostrarPares(lista):
    if len(lista) == 0:
        print("Não existem resultados.")
    else:
        for data, valor in lista:
            print(f"({dataTexto(data)}, {valor})")

tabMeteo = []
opcao = "-1"
while opcao != "0":
    menu()
    opcao = input("Selecione uma das opções: ").strip()

    if opcao == "1":
        if len(tabMeteo) == 0:
            print("A tabela meteorológica está vazia.")
        else:
            print("Temperatura média de cada dia:")
            mostrarPares(medias(tabMeteo))

    elif opcao == "2":
        if len(tabMeteo) == 0:
             print("A tabela meteorológica está vazia.")
        else:
            nomeFicheiro = input("Nome do ficheiro para guardar (ex.: meteorologia.txt): ").strip()
            guardaTabMeteo(tabMeteo, nomeFicheiro)
            print("Tabela meteorológica guardada com sucesso.")

    elif opcao == "3":
        nomeFicheiro = input("Nome do ficheiro a carregar (ex.: meteorologia.txt): ").strip()
        tabMeteo = carregaTabMeteo(nomeFicheiro)
        print("Tabela meteorológica carregada com sucesso.")
        print(f"Número de registos: {len(tabMeteo)}")

    elif opcao == "4":
        minima = minMin(tabMeteo)
        if minima is None:
            print("A tabela meteorológica está vazia.")
        else:
            print(f"A temperatura mínima mais baixa foi {minima} ºC.")

    elif opcao == "5":
        if len(tabMeteo) == 0:
            print("A tabela meteorológica está vazia.")
        else:
            print("Amplitude térmica de cada dia:")
            mostrarPares(amplTerm(tabMeteo))

    elif opcao == "6":
        resultado = maxChuva(tabMeteo)
        if resultado is None:
            print("A tabela meteorológica está vazia.")
        else:
            data, prec = resultado
            print(f"Dia com precipitação máxima: {dataTexto(data)} — {prec} mm")

    elif opcao == "7":
        p = input("Introduza o limite de precipitação p (mm): ").strip()
        p = float(p)
        print(f"Dias com precipitação superior a {p} mm:")
        mostrarPares(diasChuvosos(tabMeteo, p))

    elif opcao == "8":
        p = input("Introduza o limite de precipitação p (mm): ").strip()
        p = float(p)
        print("Maior número consecutivo de dias com precipitação abaixo do limite:", maxPeriodoCalor(tabMeteo, p))

    elif opcao == "9":
        grafTabMeteo(tabMeteo)

    elif opcao == "0":
        print("Obrigado por utilizar a aplicação.")

    else:
        print("Opção inválida. Escolha uma opção entre 0 e 9.")